# Sentiment Analysis – Amazon Alexa Reviews

Author: Phabel Antonio Lopez-Delgado, BSc.

This notebook performs **sentiment classification** on Amazon Alexa product reviews using:
- **TF‑IDF vectorization**
- **Multinomial Naïve Bayes** classifier
- **SMOTE** oversampling to handle class imbalance
- **Gradio** interactive web interface

**Dataset:** [Amazon Alexa Reviews](https://www.kaggle.com/datasets/sid321axn/amazon-alexa-reviews) (Kaggle)

**Goal:** Predict whether a review expresses positive or negative sentiment.

In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import re

# NLP & ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE

# NLTK stopwords
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Kaggle dataset loader
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Gradio for UI
import gradio as gr

# For reproducibility
np.random.seed(42)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\phabe\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
C:\Users\phabe\miniconda3\envs\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset from Kaggle
# (requires internet and kagglehub installed)
# Dataset: https://www.kaggle.com/datasets/sid321axn/amazon-alexa-reviews
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sid321axn/amazon-alexa-reviews",
    "amazon_alexa.tsv",
)

# Create sentiment label column
df['sentiment'] = df['feedback'].apply(lambda x: 'positive' if x == 1 else 'negative')

# Quick look
print("Dataset shape:", df.shape)
df.head()

100%|██████████| 503k/503k [00:00<00:00, 653kB/s]

Dataset shape: (3150, 6)


,rating,date,variation,verified_reviews,feedback,sentiment
0,5,31-Jul-18,Charcoal Fabric,Love my Echo!,1,positive
1,5,31-Jul-18,Charcoal Fabric,Loved it!,1,positive
2,4,31-Jul-18,Walnut Finish,"Sometimes while playing a game, you can answer...",1,positive
3,5,31-Jul-18,Charcoal Fabric,I have had a lot of fun with this thing. My 4 ...,1,positive
4,5,31-Jul-18,Charcoal Fabric,Music,1,positive


## Data Overview

The dataset contains Amazon Alexa reviews with:
- `verified_reviews`: the review text
- `feedback`: 1 = positive, 0 = negative
- Other columns like rating, variant, etc.

We'll focus on `verified_reviews` as the feature and `feedback` (converted to 'positive'/'negative') as the target.

In [3]:
# Check class distribution
print("=== Class distribution ===")
print(df['sentiment'].value_counts())

# Check for missing values
print("\n=== Missing values per column ===")
print(df.isnull().sum())

# Show a sample of positive and negative reviews
print("\n=== Sample positive review ===")
print(df[df['sentiment'] == 'positive']['verified_reviews'].iloc[0])
print("\n=== Sample negative review ===")
print(df[df['sentiment'] == 'negative']['verified_reviews'].iloc[0])

=== Class distribution ===
sentiment
positive    2893
negative     257
Name: count, dtype: int64

=== Missing values per column ===
rating              0
date                0
variation           0
verified_reviews    1
feedback            0
sentiment           0
dtype: int64

=== Sample positive review ===
Love my Echo!

=== Sample negative review ===
It's like Siri, in fact, Siri answers more accurately then Alexa.  I don't see a real need for it in my household, though it was a good bargain on prime day deals.


In [4]:
# Stopwords in English (keep negations)
stop_words = set(stopwords.words('english'))
negation_words = {"no", "not", "nor", "n't", "never", "none"}
stop_words = stop_words - negation_words

def clean_text(text: str) -> str:
    """
    Clean review text:
    - Lowercase
    - Remove punctuation
    - Remove stopwords (except negations)
    """
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply cleaning
df['clean_review'] = df['verified_reviews'].apply(clean_text)

# Show example
print("Original:", df['verified_reviews'].iloc[0])
print("Cleaned:", df['clean_review'].iloc[0])

Original: Love my Echo!
Cleaned: love echo


In [5]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 2520
Test samples: 630


In [6]:
vectorizer = TfidfVectorizer(max_features=5000)  # limit to top 5000 terms

X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

print("TF‑IDF shape (train):", X_train_vect.shape)

TF‑IDF shape (train): (2520, 4048)


In [7]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_vect, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", pd.Series(y_train_resampled).value_counts())

Before SMOTE:
 sentiment
positive    2314
negative     206
Name: count, dtype: int64

After SMOTE:
 sentiment
positive    2314
negative    2314
Name: count, dtype: int64


In [8]:
model = MultinomialNB()
model.fit(X_train_resampled, y_train_resampled)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [9]:
y_pred = model.predict(X_test_vect)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

    negative       0.45      0.75      0.56        51
    positive       0.98      0.92      0.95       579

    accuracy                           0.91       630
   macro avg       0.71      0.83      0.76       630
weighted avg       0.93      0.91      0.92       630


Accuracy: 0.9063


In [10]:
def predict_sentiment(user_text: str) -> str:
    """Predict sentiment of a user-provided review."""
    cleaned = clean_text(user_text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return f"Sentiment: {prediction.upper()}"

# Create Gradio interface
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Write your Alexa review here..."),
    outputs="text",
    title="Alexa Review Sentiment Analyzer",
    description="Enter a review and the model will predict if it's POSITIVE or NEGATIVE."
)

# Launch (share=True creates a public link)
interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dd973b37215657673f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Summary

- Successfully built a **sentiment classifier** for Amazon Alexa reviews.
- **TF‑IDF** vectorization + **Multinomial Naïve Bayes** with **SMOTE** achieved **~91% accuracy**.
- The **Gradio interface** allows real‑time predictions.

**Possible improvements:**
- Try other classifiers (Logistic Regression, SVM, BERT).
- Use n‑grams (unigrams + bigrams) in TF‑IDF.
- Perform more extensive hyperparameter tuning.
- Deploy the Gradio app to Hugging Face Spaces.

**Dependencies:** `pandas`, `scikit-learn`, `imbalanced-learn`, `nltk`, `kagglehub`, `gradio`.